# Phase 2: Text-Dependent Speaker Verification
## TDNN + AAM-Softmax (ArcFace) — Kaggle Version
### Before running:
- Attach Phase 1 output dataset via **+ Add Data → Your Datasets**
- Enable **GPU T4** in Accelerator settings

In [ ]:
# Step 0: Fix PyTorch CUDA version for Kaggle
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchaudio',
    '--index-url', 'https://download.pytorch.org/whl/cu121'
], check=True)
!pip install -q librosa soundfile
print('Done.')

In [ ]:
# Step 1: Imports & Config
import os, glob, json, pickle, random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.metrics import roc_curve, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.manifold import TSNE
from scipy.interpolate import interp1d
from scipy.spatial.distance import cosine

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Safe GPU check
def get_device():
    if not torch.cuda.is_available():
        print('CPU mode'); return torch.device('cpu')
    try:
        torch.zeros(1).cuda() + 1
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        return torch.device('cuda')
    except Exception as e:
        print(f'GPU failed, using CPU: {e}')
        return torch.device('cpu')

device = get_device()

# ── Auto-find Phase 1 data ────────────────────────────────────────────────────
candidates = glob.glob('/kaggle/input/**/phase1_summary.json', recursive=True)
if candidates:
    DATA_DIR   = str(Path(candidates[0]).parent)
    BASE_INPUT = str(Path(DATA_DIR).parent)
    print(f'Phase 1 data : {DATA_DIR}')
else:
    BASE_INPUT = '/kaggle/working/voice_auth'
    DATA_DIR   = f'{BASE_INPUT}/data'
    print(f'Same session : {DATA_DIR}')

# Write outputs here
OUT_DIR     = '/kaggle/working/phase2'
RESULTS_DIR = f'{OUT_DIR}/results'
MODEL_DIR   = f'{OUT_DIR}/models'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,   exist_ok=True)

with open(f'{DATA_DIR}/phase1_summary.json') as f:
    cfg = json.load(f)

SAMPLE_RATE  = cfg['sample_rate']
SPEAKER_NAME = 'abhiram'
EMB_DIM      = 256
BATCH_SIZE   = 32
EPOCHS       = 50
INPUT_DIM    = 40
MAX_FRAMES   = 300
LR           = 1e-3

GENUINE_DIR  = f'{BASE_INPUT}/data/processed/genuine'
IMPOSTOR_DIR = f'{BASE_INPUT}/data/processed/impostor'

print(f'Read : {BASE_INPUT}')
print(f'Write: {OUT_DIR}')
print(f'Genuine  files: {len(glob.glob(GENUINE_DIR+"/*.wav"))}')
print(f'Impostor files: {len(glob.glob(IMPOSTOR_DIR+"/*.wav"))}')

In [ ]:
# Step 2: Feature Extraction + Dataset
# Phase 2 uses ONLY text-dependent files (td_* prefix = phrase recordings)
def extract_mfcc(filepath, max_frames=300):
    audio, _ = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=INPUT_DIM,
                                  n_fft=512, hop_length=160, win_length=400)
    mfcc = (mfcc - mfcc.mean(axis=1, keepdims=True)) / (mfcc.std(axis=1, keepdims=True) + 1e-8)
    T    = mfcc.shape[1]
    mfcc = mfcc[:, :max_frames] if T >= max_frames else np.pad(mfcc, ((0,0),(0,max_frames-T)))
    return mfcc.T.astype(np.float32)   # (max_frames, INPUT_DIM)


class SpeakerDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records = records
        self.augment = augment
        self.le      = LabelEncoder().fit([r['speaker_id'] for r in records])
        self.num_speakers = len(self.le.classes_)

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec  = self.records[idx]
        feat = extract_mfcc(rec['filepath'])
        if self.augment:
            feat = feat + np.random.normal(0, 0.005, feat.shape).astype(np.float32)
            feat = np.roll(feat, random.randint(-10, 10), axis=0)
        label = self.le.transform([rec['speaker_id']])[0]
        return torch.FloatTensor(feat), torch.LongTensor([label])[0]


all_genuine   = glob.glob(f'{GENUINE_DIR}/*.wav')
impostor_files = glob.glob(f'{IMPOSTOR_DIR}/*.wav')

# TEXT-DEPENDENT: only phrase recordings (td_ prefix)
td_genuine = [f for f in all_genuine if Path(f).stem.startswith('td_') or
              ('td_' in Path(f).stem) or
              # fallback: if no td_ prefix exists, use all (old Phase 1 format)
              not any(Path(g).stem.startswith('td_') for g in all_genuine)]

if not td_genuine:
    td_genuine = all_genuine  # old Phase 1 format — use all
    print('WARNING: No td_ prefix found — using all genuine files (re-run Phase 1 to fix)')
else:
    ti_count = len([f for f in all_genuine if Path(f).stem.startswith('ti_')])
    print(f'Text-dependent files (td_*): {len(td_genuine)}  ← used for training')
    print(f'Text-independent (ti_*): {ti_count}  ← skipped in Phase 2')

records = []
for f in td_genuine:
    records.append({'filepath': f, 'speaker_id': SPEAKER_NAME, 'label': 'genuine'})
for f in impostor_files:
    spk = Path(f).stem.split('_')[0]
    records.append({'filepath': f, 'speaker_id': f'imp_{spk}', 'label': 'impostor'})

random.shuffle(records)
split    = int(0.8 * len(records))
train_r  = records[:split]
test_r   = records[split:]

full_ds   = SpeakerDataset(records)
shared_le = full_ds.le
NUM_SPK   = full_ds.num_speakers

train_ds = SpeakerDataset(train_r, augment=True)
test_ds  = SpeakerDataset(test_r,  augment=False)
train_ds.le = test_ds.le = shared_le
train_ds.num_speakers = test_ds.num_speakers = NUM_SPK

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Training records : {len(train_r)}')
print(f'Test records     : {len(test_r)}')
print(f'Speakers         : {NUM_SPK}')

In [ ]:
# Step 3: TDNN Model + AAM-Softmax
class TDNNLayer(nn.Module):
    def __init__(self, in_d, out_d, ctx=5, dil=1):
        super().__init__()
        self.conv = nn.Conv1d(in_d, out_d, kernel_size=ctx, dilation=dil,
                              padding=(ctx-1)*dil//2)
        self.bn   = nn.BatchNorm1d(out_d)
    def forward(self, x): return F.relu(self.bn(self.conv(x)))


class TDNN(nn.Module):
    def __init__(self, in_d=40, emb_d=256):
        super().__init__()
        self.tdnn = nn.Sequential(
            TDNNLayer(in_d, 512, 5, 1),
            TDNNLayer(512,  512, 3, 2),
            TDNNLayer(512,  512, 3, 3),
            TDNNLayer(512,  512, 1, 1),
            TDNNLayer(512, 1500, 1, 1),
        )
        self.fc1 = nn.Sequential(nn.Linear(3000, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2))
        self.fc2 = nn.Sequential(nn.Linear(512, emb_d), nn.BatchNorm1d(emb_d))

    def forward(self, x):               # (B, T, D)
        x = self.tdnn(x.transpose(1,2)) # (B, D, T)
        x = torch.cat([x.mean(2), x.std(2)], dim=1)  # stats pooling
        return F.normalize(self.fc2(self.fc1(x)), dim=1)


class AAMSoftmax(nn.Module):
    def __init__(self, emb_d, n_cls, m=0.2, s=30.0):
        super().__init__()
        self.m = m; self.s = s
        self.w = nn.Parameter(torch.FloatTensor(n_cls, emb_d))
        nn.init.xavier_uniform_(self.w)

    def forward(self, emb, labels):
        cos = F.linear(F.normalize(emb), F.normalize(self.w)).clamp(-1+1e-7, 1-1e-7)
        oh  = torch.zeros_like(cos).scatter_(1, labels.view(-1,1), 1)
        out = oh * torch.cos(torch.acos(cos) + self.m) + (1-oh) * cos
        return F.cross_entropy(self.s * out, labels)


model = TDNN(INPUT_DIM, EMB_DIM).to(device)
aam   = AAMSoftmax(EMB_DIM, NUM_SPK).to(device)
opt   = Adam(list(model.parameters()) + list(aam.parameters()), lr=LR, weight_decay=1e-4)
sched = CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Step 4: Training Loop  (with early stopping)
def run_epoch(model, aam, loader, opt=None):
    training = opt is not None
    model.train() if training else model.eval()
    total = loss_sum = 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for feats, labels in loader:
            feats, labels = feats.to(device), labels.to(device)
            if training: opt.zero_grad()
            loss = aam(model(feats), labels)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            loss_sum += loss.item() * len(labels)
            total    += len(labels)
    return loss_sum / total


# ── Early-stopping config ─────────────────────────────────────────────────────
PATIENCE   = 10   # stop if val loss doesn't improve for this many epochs
MIN_DELTA  = 1e-4 # minimum improvement to count as "better"

history  = {'train': [], 'val': []}
best_val = float('inf')
best_ep  = 0
patience_counter = 0

print(f'Training TDNN — up to {EPOCHS} epochs on {device}')
print(f'Early stopping: patience={PATIENCE}, min_delta={MIN_DELTA}')
print(f'{"Ep":>4} {"Train":>10} {"Val":>10} {"LR":>10}  {"Patience":>8}')
print('-' * 50)

for ep in range(1, EPOCHS + 1):
    tr = run_epoch(model, aam, train_loader, opt)
    vl = run_epoch(model, aam, test_loader)
    sched.step()
    history['train'].append(tr)
    history['val'].append(vl)

    improved = vl < (best_val - MIN_DELTA)
    if improved:
        best_val = vl
        best_ep  = ep
        patience_counter = 0
        torch.save(model.state_dict(), f'{MODEL_DIR}/tdnn_best.pt')
    else:
        patience_counter += 1

    if ep % 5 == 0 or ep == 1 or improved:
        marker = ' ✓' if improved else ''
        print(f'{ep:>4} {tr:>10.4f} {vl:>10.4f} '
              f'{sched.get_last_lr()[0]:>10.6f}  '
              f'{patience_counter:>8}/{PATIENCE}{marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stop at epoch {ep}  (best val={best_val:.4f} at ep {best_ep})')
        break

print(f'\nBest: epoch {best_ep}, val loss {best_val:.4f}')

In [ ]:
# Step 5: Training Curves
eps = range(1, len(history['train']) + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eps, history['train'], label='Train', color='steelblue',  lw=2)
ax.plot(eps, history['val'],   label='Val',   color='darkorange', lw=2)
ax.axvline(best_ep, color='red', ls='--', alpha=0.7, label=f'Best ({best_ep})')
ax.set_title('TDNN — AAM-Softmax Loss', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Step 6: Extract D-Vector Embeddings
model.load_state_dict(torch.load(f'{MODEL_DIR}/tdnn_best.pt', map_location=device))
model.eval()

@torch.no_grad()
def get_embedding(filepath):
    feat = extract_mfcc(filepath)
    feat = torch.FloatTensor(feat).unsqueeze(0).to(device)
    return model(feat).cpu().numpy().flatten()

embeddings, labels_list = [], []
for rec in records:
    if not os.path.exists(rec['filepath']): continue
    embeddings.append(get_embedding(rec['filepath']))
    labels_list.append(rec['label'])

embeddings = np.array(embeddings)
labels_arr = np.array(labels_list)
os.makedirs(f'{OUT_DIR}/data', exist_ok=True)
np.save(f'{OUT_DIR}/data/embeddings.npy',       embeddings)
np.save(f'{OUT_DIR}/data/embedding_labels.npy', labels_arr)
print(f'Embeddings: {embeddings.shape}')
print(f'Genuine: {(labels_arr=="genuine").sum()}  Impostor: {(labels_arr=="impostor").sum()}')

In [ ]:
# Step 7: t-SNE Plot
print('Running t-SNE...')
emb_2d = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(embeddings)
colors  = {'genuine': '#2ecc71', 'impostor': '#e74c3c'}
markers = {'genuine': 'o',       'impostor': 's'}
fig, ax = plt.subplots(figsize=(9, 7))
for lab in np.unique(labels_arr):
    m = labels_arr == lab
    ax.scatter(emb_2d[m,0], emb_2d[m,1], c=colors.get(lab,'gray'),
               marker=markers.get(lab,'o'), label=lab.capitalize(),
               alpha=0.7, s=40, edgecolors='white', linewidths=0.4)
ax.set_title('t-SNE — D-Vector Embeddings', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/tsne.png', dpi=150)
plt.show()

In [ ]:
# Step 8: Enrollment & Verification Scores
all_genuine    = glob.glob(f'{GENUINE_DIR}/*.wav')
impostor_files = glob.glob(f'{IMPOSTOR_DIR}/*.wav')
spoof_files    = glob.glob(f'{BASE_INPUT}/data/processed/spoof/*.wav')

# TEXT-DEPENDENT enrollment: only original phrase recordings (td_ prefix, no aug)
td_originals = [f for f in all_genuine
                if Path(f).stem.startswith('td_')
                and 'aug_' not in Path(f).stem]

if not td_originals:
    # Fallback for old Phase 1 format
    td_originals = [f for f in all_genuine if 'aug_' not in Path(f).stem]

print(f'Original phrase recordings : {len(td_originals)}')
print(f'Impostor files             : {len(impostor_files)}')
print(f'Spoof files                : {len(spoof_files)}')

mid          = len(td_originals) // 2
enroll_files = td_originals[:mid]
test_genuine = td_originals[mid:]

enroll_embs   = np.array([get_embedding(f) for f in enroll_files])
speaker_model = enroll_embs.mean(axis=0)
speaker_model /= (np.linalg.norm(speaker_model) + 1e-8)
print(f'\nEnrolled with {len(enroll_files)} phrase recordings')

scores, true_labels, attack_types = [], [], []

for f in test_genuine:
    scores.append(1 - cosine(speaker_model, get_embedding(f)))
    true_labels.append(1); attack_types.append('genuine')

for f in impostor_files[:100]:
    scores.append(1 - cosine(speaker_model, get_embedding(f)))
    true_labels.append(0); attack_types.append('impostor')

for f in spoof_files[:100]:
    scores.append(1 - cosine(speaker_model, get_embedding(f)))
    true_labels.append(0); attack_types.append('spoof')

scores       = np.array(scores)
true_labels  = np.array(true_labels)
attack_types = np.array(attack_types)

print(f'\nTest pairs: {len(scores)}')
for atype in ['genuine', 'impostor', 'spoof']:
    mask = attack_types == atype
    if mask.sum() > 0:
        print(f'  {atype:10s}: {scores[mask].mean():.4f} ± {scores[mask].std():.4f}')

In [ ]:
# Step 9: Metrics — EER, TAR@FAR, AUC
fpr, tpr, thresholds = roc_curve(true_labels, scores, pos_label=1)
fnr = 1 - tpr

idx       = np.nanargmin(np.abs(fpr - fnr))
eer       = (fpr[idx] + fnr[idx]) / 2 * 100
eer_thresh = thresholds[idx]
tar_1     = float(interp1d(fpr, tpr)(0.01))   * 100
tar_01    = float(interp1d(fpr, tpr)(0.001))  * 100
auc       = roc_auc_score(true_labels, scores) * 100

print('=' * 45)
print('  TEXT-DEPENDENT TDNN — RESULTS')
print('=' * 45)
print(f'  EER              : {eer:.2f}%')
print(f'  EER Threshold    : {eer_thresh:.4f}')
print(f'  TAR @ FAR=1%     : {tar_1:.2f}%')
print(f'  TAR @ FAR=0.1%   : {tar_01:.2f}%')
print(f'  AUC              : {auc:.2f}%')
print('=' * 45)

In [ ]:
# Step 10: Evaluation Plots — ROC + DET + Score Distribution
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3)

ax1 = fig.add_subplot(gs[0])
ax1.plot(fpr*100, tpr*100, color='steelblue', lw=2.5, label=f'AUC={auc:.2f}%')
ax1.fill_between(fpr*100, tpr*100, alpha=0.1, color='steelblue')
ax1.plot([0,100],[0,100],'k--', lw=1)
ax1.scatter([eer],[100-eer], color='red', s=100, zorder=5, label=f'EER={eer:.2f}%')
ax1.set_title('ROC Curve', fontweight='bold')
ax1.set_xlabel('FAR %'); ax1.set_ylabel('TAR %')
ax1.legend(); ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(gs[1])
ax2.plot(fpr*100, fnr*100, color='darkorange', lw=2.5)
ax2.scatter([eer],[eer], color='red', s=100, zorder=5, label=f'EER={eer:.2f}%')
ax2.set_xscale('log'); ax2.set_yscale('log')
ax2.set_title('DET Curve', fontweight='bold')
ax2.set_xlabel('FAR % (log)'); ax2.set_ylabel('FRR % (log)')
ax2.legend(); ax2.grid(which='both', alpha=0.3)

ax3 = fig.add_subplot(gs[2])
ax3.hist(scores[true_labels==1], bins=40, alpha=0.65, color='#2ecc71',
         label='Genuine',  density=True, edgecolor='white')
ax3.hist(scores[true_labels==0], bins=40, alpha=0.65, color='#e74c3c',
         label='Impostor', density=True, edgecolor='white')
ax3.axvline(eer_thresh, color='black', ls='--', lw=2, label=f'Thresh={eer_thresh:.3f}')
ax3.set_title('Score Distribution', fontweight='bold')
ax3.set_xlabel('Cosine Similarity'); ax3.set_ylabel('Density')
ax3.legend(); ax3.grid(alpha=0.3)

plt.suptitle('Text-Dependent TDNN — Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 11: Confusion Matrix + Threshold Sweep
pred = (scores >= eer_thresh).astype(int)
cm   = confusion_matrix(true_labels, pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(cm, display_labels=['Impostor','Genuine']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (EER threshold)', fontweight='bold')

ts = np.linspace(0, 1, 200)
far_l, frr_l = [], []
for t in ts:
    p  = (scores >= t).astype(int)
    fp = ((p==1)&(true_labels==0)).sum(); tn = ((p==0)&(true_labels==0)).sum()
    fn = ((p==0)&(true_labels==1)).sum(); tp = ((p==1)&(true_labels==1)).sum()
    far_l.append(fp/(fp+tn+1e-8)*100)
    frr_l.append(fn/(fn+tp+1e-8)*100)
axes[1].plot(ts, far_l, label='FAR %', color='#e74c3c', lw=2)
axes[1].plot(ts, frr_l, label='FRR %', color='steelblue', lw=2)
axes[1].axvline(eer_thresh, color='black', ls='--', label='EER threshold')
axes[1].set_title('FAR vs FRR vs Threshold', fontweight='bold')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Error %')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_threshold.png', dpi=150)
plt.show()

TP,FP,TN,FN = cm[1,1],cm[0,1],cm[0,0],cm[1,0]
print(f'Accuracy : {(TP+TN)/(TP+TN+FP+FN)*100:.2f}%')
print(f'Precision: {TP/(TP+FP+1e-8)*100:.2f}%')
print(f'Recall   : {TP/(TP+FN+1e-8)*100:.2f}%')
print(f'F1       : {2*TP/(2*TP+FP+FN+1e-8)*100:.2f}%')

In [ ]:
# Step 11b: Cross-Validation Plots
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3)

# ── Plot 1: ROC curves per fold ───────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
fold_colors = ['#2ecc71','#3498db','#e74c3c','#9b59b6','#f39c12']
for r in cv_rows:
    ax1.plot(r['fpr']*100, r['tpr']*100,
             color=fold_colors[r['fold']-1], lw=1.8, alpha=0.85,
             label=f"Fold {r['fold']} (AUC={r['auc']:.1f}%)")
ax1.plot([0,100],[0,100],'k--', lw=1)
ax1.set_title('ROC — All 5 Folds', fontweight='bold')
ax1.set_xlabel('FAR %'); ax1.set_ylabel('TAR %')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# ── Plot 2: EER per fold bar chart ────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
fold_nums = [r['fold'] for r in cv_rows]
eer_vals  = [r['eer']  for r in cv_rows]
auc_vals  = [r['auc']  for r in cv_rows]
x = np.arange(len(fold_nums))
w = 0.35
b1 = ax2.bar(x - w/2, eer_vals, w, label='EER %',  color='#e74c3c', edgecolor='black')
b2 = ax2.bar(x + w/2, [100 - a for a in auc_vals], w,
             label='1 - AUC %', color='#3498db', edgecolor='black')
for bar, val in zip(list(b1)+list(b2), eer_vals + [100-a for a in auc_vals]):
    if val > 0.01:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'{val:.2f}', ha='center', fontsize=8, fontweight='bold')
ax2.axhline(np.mean(eer_vals), color='red',      ls='--', lw=1.5, alpha=0.7, label=f'Mean EER={np.mean(eer_vals):.2f}%')
ax2.axhline(0,                 color='black',    ls='-',  lw=0.5)
ax2.set_xticks(x); ax2.set_xticklabels([f'Fold {i}' for i in fold_nums])
ax2.set_title('EER & (1-AUC) per Fold', fontweight='bold')
ax2.set_ylabel('Error %'); ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

# ── Plot 3: Score distributions across all folds combined ─────────────────────
ax3 = fig.add_subplot(gs[2])
all_genuine_scores  = np.concatenate([r['scores'][r['labels']==1] for r in cv_rows])
all_impostor_scores = np.concatenate([r['scores'][r['labels']==0] for r in cv_rows])
ax3.hist(all_genuine_scores,  bins=40, alpha=0.65, color='#2ecc71',
         label=f'Genuine  (n={len(all_genuine_scores)})',  density=True, edgecolor='white')
ax3.hist(all_impostor_scores, bins=40, alpha=0.65, color='#e74c3c',
         label=f'Non-genuine (n={len(all_impostor_scores)})', density=True, edgecolor='white')
mean_thresh = np.mean([cv_rows[i]['fpr'][np.nanargmin(np.abs(
    cv_rows[i]['fpr'] - (1-cv_rows[i]['tpr'])))] for i in range(len(cv_rows))])
ax3.set_title('Score Distribution (All Folds)', fontweight='bold')
ax3.set_xlabel('Cosine Similarity'); ax3.set_ylabel('Density')
ax3.legend(fontsize=9); ax3.grid(alpha=0.3)

plt.suptitle(
    f'5-Fold Cross-Validation  —  EER: {np.mean(eers):.2f}% ± {np.std(eers):.2f}%  |  '
    f'AUC: {np.mean(aucs):.2f}% ± {np.std(aucs):.2f}%',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 11a: 5-Fold Cross-Validation
# Uses ALL 6000 files — genuine (td_ original + augmented), all impostor, all spoof
import os, json, glob as _glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
from pathlib import Path as _Path
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.interpolate import interp1d
from scipy.spatial.distance import cosine

_SEED   = 42
_SR     = 16000
_MAXF   = 300
_INDIM  = 40
_EMBDIM = 256
_RESDIR = '/kaggle/working/phase2/results'
os.makedirs(_RESDIR, exist_ok=True)

# ── Find checkpoint (search multiple possible locations) ──────────────────────
_CKPT_CANDIDATES = [
    '/kaggle/working/phase2/models/tdnn_best.pt',
    '/kaggle/working/voice_auth/models/tdnn_best.pt',
    '/kaggle/working/tdnn_best.pt',
]
_CKPT = next((p for p in _CKPT_CANDIDATES if os.path.exists(p)), None)
if _CKPT is None:
    # Try a broader search
    _hits = _glob.glob('/kaggle/working/**/tdnn_best.pt', recursive=True)
    if _hits:
        _CKPT = _hits[0]

# ── Path discovery ────────────────────────────────────────────────────────────
def _find_dirs():
    for root in ['/kaggle/input', '/kaggle/working']:
        for g in _glob.glob(f'{root}/**/genuine', recursive=True):
            g = _Path(g)
            if (g.parent/'impostor').is_dir() and (g.parent/'spoof').is_dir():
                return str(g), str(g.parent/'impostor'), str(g.parent/'spoof')
    return None, None, None

if 'GENUINE_DIR' in dir() and os.path.isdir(str(GENUINE_DIR)):
    _GEN, _IMP = str(GENUINE_DIR), str(IMPOSTOR_DIR)
    _SPOOF = str(_Path(_GEN).parent / 'spoof')
else:
    _GEN, _IMP, _SPOOF = _find_dirs()
    if _GEN is None:
        raise RuntimeError('Could not find genuine/impostor/spoof dirs.')

print(f'Genuine  : {_GEN}')
print(f'Impostor : {_IMP}')
print(f'Spoof    : {_SPOOF}')
print(f'Checkpoint: {_CKPT}')

# ── Rebuild TDNN architecture ─────────────────────────────────────────────────
_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class _TDNNLayer(nn.Module):
    def __init__(self, in_d, out_d, ctx=5, dil=1):
        super().__init__()
        self.conv = nn.Conv1d(in_d, out_d, kernel_size=ctx, dilation=dil,
                              padding=(ctx-1)*dil//2)
        self.bn = nn.BatchNorm1d(out_d)
    def forward(self, x): return F.relu(self.bn(self.conv(x)))

class _TDNN(nn.Module):
    def __init__(self, in_d=40, emb_d=256):
        super().__init__()
        self.tdnn = nn.Sequential(
            _TDNNLayer(in_d, 512, 5, 1), _TDNNLayer(512, 512, 3, 2),
            _TDNNLayer(512, 512, 3, 3),  _TDNNLayer(512, 512, 1, 1),
            _TDNNLayer(512, 1500, 1, 1),
        )
        self.fc1 = nn.Sequential(nn.Linear(3000, 512), nn.BatchNorm1d(512),
                                  nn.ReLU(), nn.Dropout(0.2))
        self.fc2 = nn.Sequential(nn.Linear(512, emb_d), nn.BatchNorm1d(emb_d))
    def forward(self, x):
        x = self.tdnn(x.transpose(1, 2))
        x = torch.cat([x.mean(2), x.std(2)], dim=1)
        return F.normalize(self.fc2(self.fc1(x)), dim=1)

# ── Load model ────────────────────────────────────────────────────────────────
if 'model' in dir() and hasattr(model, 'tdnn'):
    _model = model
    print('Using model already in memory.')
elif _CKPT:
    _model = _TDNN(_INDIM, _EMBDIM).to(_device)
    _model.load_state_dict(torch.load(_CKPT, map_location=_device))
    _model.eval()
    print(f'Loaded checkpoint: {_CKPT}')
else:
    print('\n' + '='*60)
    print('ERROR: No trained model found.')
    print('You must run cells 1–7 (training) before running CV.')
    print('\nSteps:')
    print('  1. Click "Run All" OR run cells in order: 1, 2, 3, 4, 5, 6, 7')
    print('  2. Wait for training to finish (Step 4 cell)')
    print('  3. Re-run this CV cell')
    print(f'\nCheckpoint will be saved to: {_CKPT_CANDIDATES[0]}')
    print('='*60)
    raise RuntimeError('No trained model found. Run cells 1-7 first.')

# ── Feature extraction ────────────────────────────────────────────────────────
def _extract_mfcc(path):
    audio, _ = librosa.load(path, sr=_SR, mono=True)
    mfcc = librosa.feature.mfcc(y=audio, sr=_SR, n_mfcc=_INDIM,
                                  n_fft=512, hop_length=160, win_length=400)
    mfcc = (mfcc - mfcc.mean(1, keepdims=True)) / (mfcc.std(1, keepdims=True) + 1e-8)
    T = mfcc.shape[1]
    mfcc = mfcc[:, :_MAXF] if T >= _MAXF else np.pad(mfcc, ((0,0),(0,_MAXF-T)))
    return mfcc.T.astype(np.float32)

@torch.no_grad()
def _get_emb(path):
    feat = _extract_mfcc(path)
    t = torch.FloatTensor(feat).unsqueeze(0).to(_device)
    _model.eval()
    return _model(t).cpu().numpy().flatten()

# ── File lists ────────────────────────────────────────────────────────────────
td_all   = sorted([f for f in _glob.glob(f'{_GEN}/*.wav') if _Path(f).stem.startswith('td_')])
if not td_all:
    td_all = sorted(_glob.glob(f'{_GEN}/*.wav'))
imp_cv   = sorted(_glob.glob(f'{_IMP}/*.wav'))
spoof_cv = sorted(_glob.glob(f'{_SPOOF}/*.wav'))

print(f'\nGenuine: {len(td_all)}  Impostor: {len(imp_cv)}  '
      f'Spoof: {len(spoof_cv)}  Total: {len(td_all)+len(imp_cv)+len(spoof_cv)}')
if not td_all:
    raise RuntimeError(f'No WAVs in {_GEN}')

# ── Pre-compute embeddings ────────────────────────────────────────────────────
print('\nPre-computing embeddings...')
emb_cache, n_errors, first_err = {}, 0, None
for i, f in enumerate(td_all + imp_cv + spoof_cv):
    try:
        emb_cache[f] = _get_emb(f)
    except Exception as e:
        n_errors += 1
        if first_err is None:
            first_err = f'{type(e).__name__}: {e}'
    if (i+1) % 500 == 0:
        print(f'  {i+1}  ok={len(emb_cache)}  err={n_errors}')

print(f'Cached {len(emb_cache)}  errors={n_errors}')
if first_err: print(f'First error: {first_err}')
if not emb_cache:
    raise RuntimeError('All embeddings failed.')

td_originals = [f for f in td_all if 'aug_' not in _Path(f).stem]
td_augmented = [f for f in td_all if 'aug_'     in _Path(f).stem]
print(f'Originals: {len(td_originals)}  Augmented: {len(td_augmented)}')

# ── 5-Fold CV ─────────────────────────────────────────────────────────────────
N_FOLDS = min(5, len(td_originals))
kf      = KFold(n_splits=N_FOLDS, shuffle=True, random_state=_SEED)

cv_rows = []
print(f'\n{"Fold":>5} {"EER%":>8} {"AUC%":>8} {"TAR@1%":>10} '
      f'{"TAR@0.1%":>10} {"Enroll":>8} {"TestPos":>8} {"TestNeg":>8}')
print('─' * 70)

for fold, (enroll_idx, test_orig_idx) in enumerate(kf.split(td_originals)):
    enroll_files = [td_originals[i] for i in enroll_idx    if td_originals[i] in emb_cache]
    test_orig    = [td_originals[i] for i in test_orig_idx if td_originals[i] in emb_cache]
    if not enroll_files or not test_orig:
        continue

    template  = np.array([emb_cache[f] for f in enroll_files]).mean(axis=0)
    template /= (np.linalg.norm(template) + 1e-8)

    fold_scores, fold_labels = [], []
    for f in test_orig + td_augmented:
        if f in emb_cache:
            fold_scores.append(1 - cosine(template, emb_cache[f])); fold_labels.append(1)
    for f in imp_cv + spoof_cv:
        if f in emb_cache:
            fold_scores.append(1 - cosine(template, emb_cache[f])); fold_labels.append(0)

    fs = np.array(fold_scores); fl = np.array(fold_labels)
    fpr_f, tpr_f, _ = roc_curve(fl, fs, pos_label=1)
    fnr_f    = 1 - tpr_f
    ei       = np.nanargmin(np.abs(fpr_f - fnr_f))
    eer_f    = (fpr_f[ei] + fnr_f[ei]) / 2 * 100
    auc_f    = roc_auc_score(fl, fs) * 100
    interp_f = interp1d(fpr_f, tpr_f, bounds_error=False,
                        fill_value=(tpr_f[0], tpr_f[-1]))
    tar1_f   = float(interp_f(0.01))  * 100
    tar01_f  = float(interp_f(0.001)) * 100

    cv_rows.append({'fold': fold+1, 'eer': eer_f, 'auc': auc_f,
                    'tar1': tar1_f, 'tar01': tar01_f,
                    'fpr': fpr_f,   'tpr': tpr_f,
                    'scores': fs,   'labels': fl})
    print(f'{fold+1:>5} {eer_f:>8.2f} {auc_f:>8.2f} {tar1_f:>10.2f} '
          f'{tar01_f:>10.2f} {len(enroll_files):>8} '
          f'{int(fl.sum()):>8} {int((fl==0).sum()):>8}')

eers   = [r['eer']   for r in cv_rows]
aucs   = [r['auc']   for r in cv_rows]
tar1s  = [r['tar1']  for r in cv_rows]
tar01s = [r['tar01'] for r in cv_rows]

print('─' * 70)
print(f'{"Mean":>5} {np.mean(eers):>8.2f} {np.mean(aucs):>8.2f} '
      f'{np.mean(tar1s):>10.2f} {np.mean(tar01s):>10.2f}')
print(f'{"Std":>5}  {np.std(eers):>8.2f} {np.std(aucs):>8.2f} '
      f'{np.std(tar1s):>10.2f} {np.std(tar01s):>10.2f}')
print(f'\nEER : {np.mean(eers):.2f}% ± {np.std(eers):.2f}%')
print(f'AUC : {np.mean(aucs):.2f}% ± {np.std(aucs):.2f}%')

cv_summary = {
    'n_folds': N_FOLDS, 'genuine_aug_included': True,
    'total_files': len(td_all)+len(imp_cv)+len(spoof_cv),
    'eer_mean':   round(float(np.mean(eers)),  2),
    'eer_std':    round(float(np.std(eers)),   2),
    'auc_mean':   round(float(np.mean(aucs)),  2),
    'auc_std':    round(float(np.std(aucs)),   2),
    'tar1_mean':  round(float(np.mean(tar1s)), 2),
    'tar01_mean': round(float(np.mean(tar01s)),2),
    'per_fold':   [{'fold': r['fold'], 'eer': round(r['eer'],2),
                    'auc': round(r['auc'],2)} for r in cv_rows]
}
with open(f'{_RESDIR}/cv_summary.json', 'w') as f:
    json.dump(cv_summary, f, indent=2)
print(f'\nSaved: {_RESDIR}/cv_summary.json')

In [ ]:
# Step 12: Save Results
# Fallback values in case variables were lost
_emb_dim   = EMB_DIM   if 'EMB_DIM'   in dir() else 256
_epochs    = EPOCHS    if 'EPOCHS'    in dir() else 50
_best_val  = best_val  if 'best_val'  in dir() else 0.0
_best_ep   = best_ep   if 'best_ep'   in dir() else 0
_eer       = eer       if 'eer'       in dir() else 0.0
_eer_thresh= eer_thresh if 'eer_thresh' in dir() else 0.0
_tar_1     = tar_1     if 'tar_1'     in dir() else 0.0
_tar_01    = tar_01    if 'tar_01'    in dir() else 0.0
_auc       = auc       if 'auc'       in dir() else 0.0

results = {
    'model'            : 'TDNN + AAM-Softmax',
    'embedding_dim'    : _emb_dim,
    'epochs'           : _epochs,
    'best_val_loss'    : round(_best_val, 4),
    'best_epoch'       : _best_ep,
    'eer_pct'          : round(_eer, 2),
    'eer_threshold'    : round(float(_eer_thresh), 4),
    'tar_at_far_1pct'  : round(_tar_1, 2),
    'tar_at_far_01pct' : round(_tar_01, 2),
    'auc_pct'          : round(_auc, 2),
}
with open(f'{RESULTS_DIR}/phase2_results.json', 'w') as f:
    json.dump(results, f, indent=2)
with open(f'{MODEL_DIR}/enrollment.pkl', 'wb') as f:
    pickle.dump({'speaker_model': speaker_model, 'threshold': _eer_thresh}, f)

print('Phase 2 Complete!')
print('=' * 45)
for k, v in results.items():
    print(f'  {k:25s}: {v}')
print(f'\nModels  : {MODEL_DIR}')
print(f'Results : {RESULTS_DIR}')
print('\nNext: Phase 3 — Text-Independent (ECAPA-TDNN)')
